In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import napari
import colorcet as cc
import pandas as pd

import dnt

spots_directory = Path(r"C:\Tracking\BlastodermAnalysis\data\spots")
save_path = Path(r"C:\Tracking\BlastodermAnalysis\data\for_david")
save_path.mkdir(parents=True, exist_ok=True)

include = [1, 4, 6, 7, 8, 9, 10, 11, 12, 13, 14]
condition = [0, 0, 0, 0, 0, 1, 1, 2, 2, 2, 2]
earliest_frames = [25, 43, 63, 80, 75, 150, 23, 36, 17, 70, 3]

dnt.set_plot_style()

spots_dfs, metadatas, stems = dnt.load_spots_data(spots_directory, include)

df = spots_dfs[0]
cycles = [10, 11, 12, 13, 14]

print(df.columns)

In [ ]:
def calculate_tracks(df: pd.DataFrame):
    df["track_id"] = df.index
    for _frame, group in df.groupby("frame"):
        group_subset = group[group["parent_id"] > 0]
        df.loc[group_subset.index, "track_id"] = (
            group_subset["parent_id"].map(df["track_id"]).fillna(-1).astype(int)
        )

    return df

def calculate_tracklets(df: pd.DataFrame):
    df["n_children"] = 0
    df["n_children"] += df.index.map(df.groupby("parent_id")["track_id"].count())
    df["parent_n_children"] = df["parent_id"].map(df["n_children"]).fillna(0).astype(int)
    df["tracklet_id"] = df.index
    for _frame, group in df.groupby("frame"):
        group_subset = group[group["parent_n_children"] == 1]
        df.loc[group_subset.index, "tracklet_id"] = (
            group_subset["parent_id"].map(df["tracklet_id"]).fillna(-1).astype(int)
        )

    return df

spots_dfs[5] = calculate_tracks(spots_dfs[5])
spots_dfs[5] = calculate_tracklets(spots_dfs[5])

In [ ]:
spots_dfs[6].groupby("tracklet_id")["cycle"].min().describe()

In [ ]:
export_columns = ['frame', 'time_since_nc11', 'cycle', 'z', 'y', 'x', 'tracklet_id', 'parent_id', 'track_id', 'AP']

for k, df in enumerate(spots_dfs):
    stem = stems[k]
    c = condition[k]
    if c == 1:
        df["time_since_nc11"] /= 60

    earliest_frame = earliest_frames[k]
    df = df[df["frame"] >= earliest_frame]

    df = df[export_columns]

    df.to_csv(save_path / "everything" / f"{k}_{stem}.csv", index=True)

In [ ]:
spots_dfs[6]

In [ ]:
for k, df in enumerate(spots_dfs):
    stem = stems[k]
    earliest_frame = earliest_frames[k]
    df = df[df["frame"] >= earliest_frame]
    df = df[~df["time_since_nc11"].isna()]
    fig, ax = plt.subplots(figsize=(10, 6))
    for cycle in cycles:

        cycle_df = df[df["cycle"] == cycle].copy()

        start_times = cycle_df.groupby("tracklet_id")["time_since_nc11"].min()
        sns.kdeplot(start_times, ax=ax, label=f"Cycle {cycle}", fill=True, common_norm=False)
        last_start = np.quantile(start_times, 0.98)

        end_times = cycle_df.groupby("tracklet_id")["time_since_nc11"].max()
        sns.kdeplot(end_times, ax=ax, label=f"Cycle {cycle} End", fill=True, common_norm=False, linestyle="--")
        first_end = np.quantile(end_times, 0.02)

        print(f"Cycle {cycle} for {stem}: {last_start:.2f} to {first_end:.2f} minutes")
    ax.set_title(stem)
    plt.show()